# Diagnóstico de Negocio: ¿qué SKUs merecen un modelo dedicado?
## Segmentación del portafolio en *tiers* de forecasting

> **Fase 1 — Video 7** | Análisis Exploratorio de Datos
> Dataset: Histórico de Ventas Sintético

---

### El error que arruina proyectos de forecasting: tratar los 50 SKUs igual

Antes de modelar nada, hay una decisión de negocio: **¿dónde vale la pena invertir un modelo dedicado y
dónde no?** Un SKU estrella que factura millones y vende todos los días **no** se modela igual que una
refacción que vende 3 veces al mes. Ponerle un SARIMAX a la refacción es tan absurdo como ponerle un
promedio simple al producto estrella.

Este video construye un **diagnóstico cuantitativo** que clasifica cada SKU en un *tier* de forecasting,
usando tres ejes objetivos:

| Eje | Métrica | Pregunta |
|---|---|---|
| **Valor** | Contribución al revenue (ABC / Pareto) | ¿Cuánto duele equivocarse en este SKU? |
| **Patrón de demanda** | Clasificación SBC (ADI × CV²) | ¿Es regular o intermitente? |
| **Predictibilidad** | Error de un baseline estacional (WAPE) | ¿Qué tan pronosticable es su historia? |

Al final saldrás con el **catálogo etiquetado**: qué SKUs van a un modelo dedicado (SARIMAX), cuáles a un
modelo global (XGBoost) y cuáles a métodos de demanda intermitente (Croston/TSB, → Video 14). Este video
**justifica todo lo que viene en la Fase 3**.

### Ruta del notebook
1. Librerías y carga (ventas + catálogo)
2. Eje 1 — Contribución al revenue: análisis ABC (Pareto)
3. Eje 2 — Patrón de demanda: clasificación SBC (ADI × CV²)
4. Eje 3 — Predictibilidad: skill contra un baseline estacional
5. La matriz de decisión: asignación de *tiers*
6. El catálogo clasificado + validación contra la verdad de campo
7. Conclusiones y puente a la Fase 2 / Fase 3

---
## 1. Librerías y carga de datos

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, PercentFormatter
from pathlib import Path

plt.rcParams.update({
    'figure.facecolor': '#F8FAFC', 'axes.facecolor': '#F8FAFC',
    'axes.grid': True, 'grid.color': '#E2E8F0',
    'grid.linewidth': 0.8, 'font.size': 11,
})
BLUE, RED, GREEN, PURPLE, ORANGE = '#2563EB','#DC2626','#16A34A','#7C3AED','#EA580C'
money_fmt = FuncFormatter(lambda x, _: f'${x/1e6:.1f}M')
print('✅ Librerías cargadas')

In [ ]:
def find_csv(filename, max_levels=4):
    base = Path().resolve()
    for _ in range(max_levels):
        candidate = base / 'output' / filename
        if candidate.exists():
            return candidate
        base = base.parent
    raise FileNotFoundError(f"No se encontró '{filename}'. Corre main.py primero.")

sales_path = find_csv('sales_history.csv')
df = pd.read_csv(sales_path, parse_dates=['date']).sort_values('date')
catalog = pd.read_csv(find_csv('sku_catalog.csv')).set_index('sku_id')

# Matriz de demanda semanal por SKU (unidades). Semanas sin venta = 0.
weekly = (
    df.groupby(['sku_id', pd.Grouper(key='date', freq='W-MON')])['units_sold']
      .sum()
      .unstack('sku_id')
      .fillna(0)
      .sort_index()
)
revenue = df.groupby('sku_id')['revenue'].sum()

print(f'✅ {df["sku_id"].nunique()} SKUs | {weekly.shape[0]} semanas por SKU')
print(f'   Revenue total del portafolio: ${revenue.sum():,.0f}')

---
## 2. Eje 1 — Contribución al revenue (análisis ABC)

La ley de Pareto casi siempre aplica: **una minoría de SKUs genera la mayoría del revenue**. El análisis
ABC ordena los productos por facturación acumulada y los parte en clases:

| Clase | Revenue acumulado | Interpretación |
|---|---|---|
| **A** | hasta 80% | Los pocos vitales — un error aquí cuesta caro |
| **B** | 80–95% | Importancia media |
| **C** | 95–100% | La cola larga — muchos SKUs, poco revenue cada uno |

El punto clave para forecasting: **el esfuerzo de modelado debe seguir al dinero**.

In [ ]:
rev_sorted = revenue.sort_values(ascending=False)
cum_share  = rev_sorted.cumsum() / rev_sorted.sum()
abc = pd.cut(cum_share, [0, 0.80, 0.95, 1.0], labels=['A', 'B', 'C'])
abc.name = 'abc'

fig, ax1 = plt.subplots(figsize=(14, 5))
colors_abc = {'A': GREEN, 'B': ORANGE, 'C': RED}
bar_colors = [colors_abc[c] for c in abc.values]
ax1.bar(range(len(rev_sorted)), rev_sorted.values, color=bar_colors, width=0.9)
ax1.yaxis.set_major_formatter(money_fmt)
ax1.set_xlabel('SKUs (ordenados por revenue)')
ax1.set_ylabel('Revenue total')
ax1.set_title('Análisis ABC / Pareto del portafolio', fontsize=13, fontweight='bold')

ax2 = ax1.twinx()
ax2.plot(range(len(cum_share)), cum_share.values, color=BLUE, linewidth=2.2, marker='o', ms=3)
ax2.axhline(0.80, color=GREEN, linestyle='--', linewidth=1)
ax2.axhline(0.95, color=ORANGE, linestyle='--', linewidth=1)
ax2.yaxis.set_major_formatter(PercentFormatter(1.0))
ax2.set_ylabel('Revenue acumulado')
ax2.grid(False)

handles = [plt.Rectangle((0, 0), 1, 1, color=colors_abc[c]) for c in ['A', 'B', 'C']]
ax1.legend(handles, [f'Clase {c}' for c in ['A', 'B', 'C']], loc='center right')
plt.tight_layout()
plt.show()

counts = abc.value_counts().reindex(['A', 'B', 'C'])
rev_by_abc = revenue.groupby(abc).sum().reindex(['A', 'B', 'C'])
print('Clase  SKUs   % SKUs   % Revenue')
for c in ['A', 'B', 'C']:
    print(f'  {c}    {counts[c]:>3}    {counts[c]/len(abc):>6.0%}    {rev_by_abc[c]/revenue.sum():>7.0%}')

---
## 3. Eje 2 — Patrón de demanda (clasificación SBC)

El revenue no basta: un SKU puede facturar poco **y además** tener una demanda imposible de pronosticar
con métodos clásicos. La taxonomía de **Syntetos–Boylan–Croston (SBC)** clasifica cada serie con dos
números:

- **ADI** (*Average Demand Interval*): cada cuántos períodos, en promedio, hay una venta.
  `ADI = nº de períodos / nº de períodos con venta`. ADI alto ⇒ muchos ceros ⇒ intermitente.
- **CV²**: varianza relativa del **tamaño** de las ventas no nulas. CV² alto ⇒ tamaños erráticos.

Los umbrales clásicos (ADI = 1.32, CV² = 0.49) parten el plano en cuatro cuadrantes:

| | CV² bajo (tamaño estable) | CV² alto (tamaño errático) |
|---|---|---|
| **ADI bajo (frecuente)** | 🟢 **Smooth** — ARIMA/ETS funcionan | 🟠 **Erratic** — ML global |
| **ADI alto (con ceros)** | 🔵 **Intermittent** — Croston | 🔴 **Lumpy** — TSB/lo más difícil |

In [ ]:
def classify_sbc(series):
    s  = series.values
    nz = s[s > 0]
    n_periods = len(s)
    n_demand  = int((s > 0).sum())
    if n_demand == 0:
        return np.nan, np.nan, 'Sin ventas'
    adi = n_periods / n_demand
    cv2 = (nz.std() / nz.mean()) ** 2 if n_demand > 1 else 0.0
    if   adi < 1.32 and cv2 < 0.49: klass = 'Smooth'
    elif adi < 1.32 and cv2 >= 0.49: klass = 'Erratic'
    elif adi >= 1.32 and cv2 < 0.49: klass = 'Intermittent'
    else: klass = 'Lumpy'
    return adi, cv2, klass

sbc = pd.DataFrame(
    {sku: classify_sbc(weekly[sku]) for sku in weekly.columns},
    index=['adi', 'cv2', 'sbc_class'],
).T
sbc[['adi', 'cv2']] = sbc[['adi', 'cv2']].astype(float)

class_colors = {'Smooth': GREEN, 'Erratic': ORANGE, 'Intermittent': BLUE, 'Lumpy': RED}

fig, ax = plt.subplots(figsize=(11, 7))
for k, c in class_colors.items():
    m = sbc['sbc_class'] == k
    ax.scatter(sbc.loc[m, 'adi'], sbc.loc[m, 'cv2'], s=70, color=c, alpha=0.8,
               edgecolor='white', linewidth=0.6, label=k)
ax.axvline(1.32, color='black', linestyle='--', linewidth=1)
ax.axhline(0.49, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('ADI  →  más ceros (intermitencia)')
ax.set_ylabel('CV²  →  tamaño más errático')
ax.set_title('Clasificación SBC del portafolio (demanda semanal)', fontsize=13, fontweight='bold')
ax.legend(title='Patrón')
# etiquetas de cuadrante
ax.text(1.03, 0.05, 'Smooth', color=GREEN, fontweight='bold')
ax.text(sbc['adi'].max()*0.7, 0.05, 'Intermittent', color=BLUE, fontweight='bold')
ax.text(1.03, sbc['cv2'].max()*0.92, 'Erratic', color=ORANGE, fontweight='bold')
ax.text(sbc['adi'].max()*0.7, sbc['cv2'].max()*0.92, 'Lumpy', color=RED, fontweight='bold')
plt.tight_layout()
plt.show()

print('Distribución por patrón de demanda:')
for k, v in sbc['sbc_class'].value_counts().items():
    print(f'  {k:<13}: {v:>3} SKUs')

---
## 4. Eje 3 — Predictibilidad (skill contra un baseline)

Antes de gastar en un modelo sofisticado, mide qué tan lejos llega lo **barato**. Usamos un **baseline
estacional** (*seasonal naive*): "la demanda de esta semana será la misma que hace 52 semanas". Medimos su
error con **WAPE** (*Weighted Absolute Percentage Error*), robusto ante ceros — a diferencia del MAPE, que
explota cuando hay demanda cero:

$$\text{WAPE} = \frac{\sum_t |y_t - \hat y_t|}{\sum_t |y_t|}$$

WAPE bajo ⇒ la historia ya predice bien el futuro ⇒ un modelo dedicado tiene poco que ganar.
WAPE alto (típico de intermitentes) ⇒ ni el baseline funciona ⇒ hace falta un método especializado.

In [ ]:
def snaive_wape(series, m=52, horizon=52):
    s = series.dropna()
    if len(s) <= m + horizon // 2:
        return np.nan
    y    = s.iloc[-horizon:]
    yhat = s.shift(m).iloc[-horizon:]
    denom = y.abs().sum()
    return float((y - yhat).abs().sum() / denom) if denom > 0 else np.nan

wape = weekly.apply(snaive_wape).rename('wape')

fig, ax = plt.subplots(figsize=(12, 5))
order = wape.sort_values()
colors_w = [class_colors[sbc.loc[sku, 'sbc_class']] for sku in order.index]
ax.bar(range(len(order)), order.values, color=colors_w, width=0.9)
ax.axhline(order.median(), color='black', linestyle='--', linewidth=1,
           label=f'Mediana WAPE: {order.median():.2f}')
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel('SKUs (ordenados por WAPE del baseline estacional)')
ax.set_ylabel('WAPE seasonal-naive')
ax.set_title('Predictibilidad: error del baseline por SKU (color = patrón SBC)',
             fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print(f'WAPE del baseline estacional — mediana: {wape.median():.1%}')
print('Los SKUs con WAPE más alto (peor pronosticables):')
for sku, w in wape.sort_values(ascending=False).head(5).items():
    print(f'  {sku}: WAPE {w:.0%}  ({sbc.loc[sku, "sbc_class"]})')

---
## 5. La matriz de decisión: asignación de *tiers*

Cruzamos los tres ejes en una regla explícita y accionable. La lógica: **el patrón de demanda decide el
tipo de modelo; el valor decide cuánto lo mimas.**

| Tier | Criterio | Modelo recomendado | Por qué |
|---|---|---|---|
| **A — Dedicado** | Clase A (revenue) **y** demanda *Smooth* | **SARIMAX** por SKU + variables exógenas | Alto valor y pronosticable: vale un modelo a la medida |
| **B — Global** | El resto de la demanda regular/errática | **XGBoost / LightGBM** global | Un modelo entrenado sobre todos, con features compartidos |
| **C — Intermitente** | Demanda *Intermittent* o *Lumpy* | **Croston / TSB / ADIDA** | ARIMA y ML colapsan con tantos ceros — necesitan métodos propios |

> Además, las **categorías** y el total se modelan de forma **jerárquica** con reconciliación (bottom-up /
> MinT) para que los pronósticos cuadren entre niveles — el tema del Video 16.

In [ ]:
diag = pd.DataFrame({'revenue': revenue}).join(abc).join(sbc).join(wape)
diag['rev_share'] = diag['revenue'] / diag['revenue'].sum()

def assign_tier(row):
    if row['sbc_class'] in ('Intermittent', 'Lumpy'):
        return 'C · Intermitente (Croston/TSB)'
    if row['abc'] == 'A' and row['sbc_class'] == 'Smooth':
        return 'A · Dedicado (SARIMAX)'
    return 'B · Global (XGBoost)'

diag['tier'] = diag.apply(assign_tier, axis=1)
tier_order = ['A · Dedicado (SARIMAX)', 'B · Global (XGBoost)', 'C · Intermitente (Croston/TSB)']
tier_colors = {tier_order[0]: GREEN, tier_order[1]: ORANGE, tier_order[2]: RED}

summary = diag.groupby('tier').agg(
    skus=('revenue', 'size'),
    pct_skus=('revenue', lambda s: len(s) / len(diag)),
    pct_revenue=('revenue', lambda s: s.sum() / diag['revenue'].sum()),
    wape_medio=('wape', 'median'),
).reindex(tier_order)

print('═' * 64)
print('  PORTAFOLIO SEGMENTADO EN TIERS DE FORECASTING')
print('═' * 64)
print(f'  {"Tier":<34}{"SKUs":>5}{"% SKUs":>9}{"% Rev":>8}{"WAPE":>8}')
print('  ' + '─' * 62)
for t in tier_order:
    r = summary.loc[t]
    print(f'  {t:<34}{int(r.skus):>5}{r.pct_skus:>9.0%}{r.pct_revenue:>8.0%}{r.wape_medio:>8.0%}')
print('═' * 64)
print('  Lectura: el grueso del REVENUE se concentra en pocos SKUs (tier A),')
print('  mientras que los intermitentes (tier C) son MUCHOS SKUs pero POCO dinero.')

In [ ]:
# Vista integradora: ADI × CV² coloreado por tier, tamaño = revenue
fig, ax = plt.subplots(figsize=(12, 7))
for t in tier_order:
    m = diag['tier'] == t
    ax.scatter(diag.loc[m, 'adi'], diag.loc[m, 'cv2'],
               s=40 + 3000 * diag.loc[m, 'rev_share'], color=tier_colors[t],
               alpha=0.75, edgecolor='white', linewidth=0.7, label=t)
ax.axvline(1.32, color='black', linestyle='--', linewidth=1)
ax.axhline(0.49, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('ADI  →  intermitencia')
ax.set_ylabel('CV²  →  tamaño errático')
ax.set_title('El portafolio en una imagen (tamaño = participación en revenue)',
             fontsize=13, fontweight='bold')
ax.legend(title='Tier', loc='upper right')
plt.tight_layout()
plt.show()

---
## 6. El catálogo clasificado + validación contra la verdad de campo

Como generamos el dataset, sabemos qué SKUs son de verdad **intermitentes** (columna `demand_profile` del
catálogo). Igual que en el Video 6, aprovechamos esa verdad para **auditar** si nuestro diagnóstico SBC
—que solo miró la serie de ventas— recuperó el diseño real.

In [ ]:
diag = diag.join(catalog['demand_profile'])

# Cruce: ¿la clase SBC (derivada de los datos) coincide con el perfil real?
cross = pd.crosstab(diag['demand_profile'], diag['sbc_class'])
print('Validación — perfil REAL (filas) vs clase SBC detectada (columnas):')
print(cross.to_string())
print()

recovered = diag.loc[diag['demand_profile'] == 'Intermitente', 'sbc_class'] \
                .isin(['Intermittent', 'Lumpy', 'Erratic']).mean()
print(f'De los SKUs realmente intermitentes, {recovered:.0%} cayeron fuera de "Smooth"')
print('→ el diagnóstico basado solo en la serie separa bien los slow-movers.')

# Catálogo etiquetado (muestra por tier)
cols = ['revenue', 'rev_share', 'abc', 'adi', 'cv2', 'sbc_class', 'wape', 'demand_profile', 'tier']
tabla = diag[cols].sort_values(['tier', 'revenue'], ascending=[True, False])
print('\nMuestra del catálogo clasificado (2 SKUs por tier):')
muestra = tabla.groupby('tier', group_keys=False).head(2)
with pd.option_context('display.width', 140, 'display.max_columns', None):
    print(muestra.round({'rev_share': 3, 'adi': 2, 'cv2': 2, 'wape': 2}).to_string())

# Persistimos el catálogo etiquetado para las siguientes fases
out = find_csv('sku_catalog.csv').parent / 'sku_tiers.csv'
tabla.to_csv(out)
print(f'\nCatálogo etiquetado guardado en: {out}')

---
## 7. Conclusiones y puente a la Fase 2

### Las reglas que te llevas

1. **No modeles a ciegas todo el portafolio igual.** Primero segmenta: el esfuerzo de modelado debe
   seguir al **dinero** (ABC) y al **patrón de demanda** (SBC).
2. **El patrón decide el *tipo* de modelo; el valor decide cuánto lo mimas.** Demanda intermitente ⇒
   Croston/TSB, no ARIMA ni XGBoost — por más caro que sea el SKU.
3. **Mide siempre contra un baseline.** Si el *seasonal naive* ya da un WAPE bajo, un modelo sofisticado
   tiene poco que ganar (lo formalizamos en los Videos 11 y 18).
4. **WAPE, no MAPE, cuando hay ceros.** El MAPE es indefinido/explosivo con demanda cero — justo los SKUs
   intermitentes.
5. **La cola larga existe:** muchos SKUs de tier C aportan poco revenue. No los ignores, pero tampoco les
   dediques un SARIMAX individual.

### El mapa de lo que viene

| Tier | Se modela en… |
|---|---|
| **A · Dedicado** | Video 12 (SARIMA/SARIMAX) — con las exógenas de la Fase 2 |
| **B · Global** | Video 15 (XGBoost / LightGBM como regresión) |
| **C · Intermitente** | Video 14 (Croston, TSB, ADIDA) |
| **Categorías / total** | Video 16 (forecasting jerárquico + reconciliación MinT) |

El catálogo etiquetado (`output/sku_tiers.csv`) es el plano que guiará la Fase 3. Pero antes de modelar,
necesitamos construir las **variables predictoras**.

---

### Próximo video
**Video 8 — Features de calendario: festivos, quincenas y eventos del retail mexicano**
Arrancamos la Fase 2 (Feature Engineering) convirtiendo el calendario en señales: días festivos,
quincenas, Buen Fin, Hot Sale y fin de mes — el tipo de variable exógena que alimenta al tier A.